8. Business insights

Translate results into decision-support tables and intervention priorities.

In [5]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

import pandas as pd
import joblib
PRED, OUT = ROOT/'results/predictions/holdout_predictions_new_features_addd.csv', ROOT/'results/reports'
OUT.mkdir(parents=True, exist_ok=True)
p = pd.read_csv(PRED)
for col in ['project','type','task_group','season','location', 'to_package']:
    if col in p:
        table = p.groupby(col).agg(
            tasks=('actual_overdue','size'), 
            actual_overdue_rate=('actual_overdue','mean'), 
            mean_predicted_risk=('overdue_risk_probability','mean')
        )

        table = table.sort_values('mean_predicted_risk', ascending=False)
        table.to_csv(OUT/f'dashboard_{col}.csv')


In [6]:
with open(OUT/'management_recommendations.md', 'w', encoding='utf-8') as f:
    f.write('''# Operational & Decision-Support Recommendations

## 1. Targeted Resource Allocation by Task Domain
* **Focus Areas:** Direct engineering and oversight resources to **Design Team** and **Quality** tasks, which exhibit observed failure/overdue rates above **31%** and predicted risks near **55–59%**.
* **Process Streamlining:** Reduce administrative gating on low-risk operational categories such as **Safety**, which consistently demonstrate a near-zero overdue rate (0.02%).

## 2. Risk-Tiered Action Framework (Triage System)
Implement an automated thresholding policy in the Project Management Information System (PMIS):
* **High Risk (Probability > 0.70):** Trigger an immediate review by the Project Manager within 24 hours to re-assign resources or adjust deadlines.
* **Medium Risk (0.30 <= Probability <= 0.70):** Flag for the weekly site coordination meeting.
* **Low Risk (Probability < 0.30):** Standard operational processing without intervention.

## 3. Phased Deployment Strategy
* Initiate system integration on critical project sites—specifically **Project 1338** and **Project 1335**—which exhibit the highest average predicted overdue risks (**26.7%** and **23.7%** respectively).
* Use these projects as baseline testbeds before enterprise-wide rollout to low-risk project domains (e.g., Projects 1340 and 1345).

## 4. Key Operational Feature Drivers (Feature Engineering Insights)
Recent model enhancement through advanced spatial and historical feature engineering yielded critical drivers for risk prediction:
* **Spatial Task Density (`tasks_in_same_location`):** Identified as the **#2 most impactful feature** globally. A high concentration of tasks in a single location acts as a primary indicator of site congestion and work bottlenecks, driving individual risk scores significantly higher (+2.58 SHAP impact in peak cases).
* **Package Delay History (`package_overdue_rate`):** Ranked as the **#3 feature overall**. Serves as a strong structural risk indicator for contractors or work packages; packages with historical delay patterns drastically increase the probability of subsequent task delays (+1.70 SHAP contribution).

## 5. Explainable AI (XAI) & Model Stability Improvement
* **False Positive Reduction:** Incorporating spatial and package-level metrics increased target class Precision from **51.26% to 56.27% (+5.01%)**, substantially reducing false alarms for site supervisors. Overall model F1-Score reached **0.696** with an Accuracy of **94.5%**.
* **Balanced Decision Boundary:** Feature engineering successfully decoupled model predictions from over-reliance on single categorical features (e.g., `Safety`), balancing risk signals across spatial congestion and contractor history for enhanced robustness on unseen data.
* **Dashboard Integration:** Present the top-3 SHAP feature contributions (highlighting spatial density and historical package risk alongside task domain) for every high-risk alert to ensure field engineers act on actionable context rather than a black-box score.
''')

print('Dashboard-ready tables and updated data-driven recommendations saved to results/reports/.')

Dashboard-ready tables and updated data-driven recommendations saved to results/reports/.
